<a href="https://colab.research.google.com/github/Manjushree-2007/Manju/blob/main/Study_mate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# StudyMate - AI Academic Assistant with IBM Granite
# Run this in Google Colab

# Install required packages
!pip install -q gradio pymupdf sentence-transformers faiss-cpu huggingface_hub

import gradio as gr
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from huggingface_hub import InferenceClient
import os
from typing import List, Tuple
import re

# Initialize models
print("Loading models...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
client = InferenceClient(model="ibm-granite/granite-3.2-2b-instruct")

# Global variables
vector_store = None
text_chunks = []
chunk_metadata = []

def extract_text_from_pdf(pdf_file) -> str:
    """Extract text from a PDF file."""
    try:
        doc = fitz.open(pdf_file.name)
        text = ""
        for page_num, page in enumerate(doc):
            page_text = page.get_text()
            text += f"\n--- Page {page_num + 1} ---\n{page_text}"
        doc.close()
        return text
    except Exception as e:
        return f"Error extracting text: {str(e)}"

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> List[Tuple[str, dict]]:
    """Split text into overlapping chunks with metadata."""
    # Clean text
    text = re.sub(r'\s+', ' ', text).strip()

    chunks_with_metadata = []
    words = text.split()

    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        chunk = ' '.join(chunk_words)

        # Extract page number if available
        page_match = re.search(r'--- Page (\d+) ---', chunk)
        page_num = page_match.group(1) if page_match else "Unknown"

        # Clean the chunk from page markers
        chunk = re.sub(r'--- Page \d+ ---', '', chunk).strip()

        if len(chunk) > 50:  # Only keep meaningful chunks
            metadata = {
                'page': page_num,
                'chunk_id': len(chunks_with_metadata)
            }
            chunks_with_metadata.append((chunk, metadata))

    return chunks_with_metadata

def create_vector_store(chunks_with_metadata: List[Tuple[str, dict]]):
    """Create FAISS vector store from text chunks."""
    global vector_store, text_chunks, chunk_metadata

    text_chunks = [chunk for chunk, _ in chunks_with_metadata]
    chunk_metadata = [meta for _, meta in chunks_with_metadata]

    # Generate embeddings
    embeddings = embedder.encode(text_chunks, convert_to_numpy=True)

    # Create FAISS index
    dimension = embeddings.shape[1]
    vector_store = faiss.IndexFlatL2(dimension)
    vector_store.add(embeddings.astype('float32'))

    return f"✅ Vector store created with {len(text_chunks)} chunks"

def process_pdfs(pdf_files):
    """Process uploaded PDF files."""
    if not pdf_files:
        return "❌ Please upload at least one PDF file"

    try:
        all_text = ""
        for pdf_file in pdf_files:
            text = extract_text_from_pdf(pdf_file)
            all_text += f"\n\n{'='*50}\nDocument: {pdf_file.name}\n{'='*50}\n{text}"

        # Chunk the text
        chunks_with_meta = chunk_text(all_text)

        # Create vector store
        result = create_vector_store(chunks_with_meta)

        return f"{result}\n📄 Processed {len(pdf_files)} PDF file(s)\n💡 You can now ask questions!"

    except Exception as e:
        return f"❌ Error processing PDFs: {str(e)}"

def retrieve_relevant_chunks(query: str, top_k: int = 3) -> List[Tuple[str, dict]]:
    """Retrieve most relevant chunks for a query."""
    if vector_store is None or len(text_chunks) == 0:
        return []

    # Encode query
    query_embedding = embedder.encode([query], convert_to_numpy=True)

    # Search in FAISS
    distances, indices = vector_store.search(query_embedding.astype('float32'), top_k)

    # Retrieve chunks with metadata
    results = []
    for idx in indices[0]:
        if idx < len(text_chunks):
            results.append((text_chunks[idx], chunk_metadata[idx]))

    return results

def generate_answer(query: str, context_chunks: List[Tuple[str, dict]]) -> str:
    """Generate answer using IBM Granite model."""
    if not context_chunks:
        return "❌ No relevant information found. Please upload PDF files first."

    # Prepare context
    context = "\n\n".join([
        f"[Page {meta['page']}]: {chunk}"
        for chunk, meta in context_chunks
    ])

    # Create prompt
    prompt = f"""You are StudyMate, an AI academic assistant. Answer the student's question based on the provided context from their study materials.

Context from study materials:
{context}

Student's Question: {query}

Instructions:
- Provide a clear, accurate answer based on the context
- Reference page numbers when relevant
- If the context doesn't contain enough information, say so
- Be concise but informative

Answer:"""

    try:
        # Generate response using IBM Granite
        response = ""
        for message in client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            stream=True,
        ):
            response += message.choices[0].delta.content or ""

        # Add source references
        pages = list(set([meta['page'] for _, meta in context_chunks]))
        source_info = f"\n\n📚 **Sources:** Pages {', '.join(pages)}"

        return response.strip() + source_info

    except Exception as e:
        return f"❌ Error generating answer: {str(e)}\n\nNote: Make sure you have access to the model or set HF_TOKEN if required."

def answer_question(query: str, history):
    """Main function to answer questions."""
    if not query.strip():
        return history + [("", "Please enter a question.")]

    if vector_store is None:
        return history + [(query, "❌ Please upload and process PDF files first before asking questions.")]

    # Retrieve relevant chunks
    relevant_chunks = retrieve_relevant_chunks(query, top_k=3)

    # Generate answer
    answer = generate_answer(query, relevant_chunks)

    # Update history
    history = history + [(query, answer)]
    return history

def clear_data():
    """Clear all stored data."""
    global vector_store, text_chunks, chunk_metadata
    vector_store = None
    text_chunks = []
    chunk_metadata = []
    return "✅ All data cleared. Upload new PDFs to start fresh.", []

# Create Gradio Interface
with gr.Blocks(title="StudyMate - AI Academic Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 📚 StudyMate - AI Academic Assistant
    ### Powered by IBM Granite 3.2-2B-Instruct

    Upload your study materials (PDFs) and ask questions in natural language.
    StudyMate will provide contextual answers with page references.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Upload Documents")
            pdf_input = gr.File(
                label="Upload PDF Files",
                file_types=[".pdf"],
                file_count="multiple"
            )
            process_btn = gr.Button("🔄 Process PDFs", variant="primary")
            process_output = gr.Textbox(
                label="Processing Status",
                lines=3,
                interactive=False
            )
            clear_btn = gr.Button("🗑️ Clear All Data", variant="stop")

            gr.Markdown("""
            ### ℹ️ How to Use:
            1. Upload one or more PDF files
            2. Click "Process PDFs"
            3. Wait for processing to complete
            4. Ask questions in the chat
            """)

        with gr.Column(scale=2):
            gr.Markdown("### 💬 Ask Questions")
            chatbot = gr.Chatbot(
                label="Chat with Your Study Materials",
                height=500,
                show_copy_button=True
            )

            with gr.Row():
                question_input = gr.Textbox(
                    label="Your Question",
                    placeholder="e.g., What is the main concept explained in chapter 3?",
                    lines=2,
                    scale=4
                )
                submit_btn = gr.Button("🚀 Ask", variant="primary", scale=1)

            gr.Examples(
                examples=[
                    "Summarize the main concepts from this document",
                    "What are the key definitions mentioned?",
                    "Explain the methodology discussed in the paper",
                    "What are the main conclusions?",
                ],
                inputs=question_input,
                label="Example Questions"
            )

    # Event handlers
    process_btn.click(
        fn=process_pdfs,
        inputs=[pdf_input],
        outputs=[process_output]
    )

    submit_btn.click(
        fn=answer_question,
        inputs=[question_input, chatbot],
        outputs=[chatbot]
    ).then(
        lambda: "",
        outputs=[question_input]
    )

    question_input.submit(
        fn=answer_question,
        inputs=[question_input, chatbot],
        outputs=[chatbot]
    ).then(
        lambda: "",
        outputs=[question_input]
    )

    clear_btn.click(
        fn=clear_data,
        outputs=[process_output, chatbot]
    )

# Launch the app
print("\n" + "="*50)
print("🚀 Launching StudyMate...")
print("="*50)
demo.launch(debug=True, share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 39.6 MB/s eta 0:00:00
Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipython-input-2112428621.py:234: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(



🚀 Launching StudyMate...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d0693b3e8f31320fce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
